# Running Inference with the Mistral 7B Model


### For fine-tuning Llama, a GPU instance is essential. Follow the directions below:

1. Go to *Runtime*.
2. Select *Change Runtime Type*.
3. Choose *T4 GPU*.


### Installs the *transformers* library directly from the HuggingFace GitHub repository and other required libraries

- *accelerate*: use hardware accelerators like GPUs and TPUs more efficiently.
- *bitsandbytes*: fast gradient compression
- *sentencepiece*: used for DL-based text processing


In [1]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-axuwycr0
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-axuwycr0
  Resolved https://github.com/huggingface/transformers to commit aa1c36f1a9f454e69c4eac83071ced235942c7ed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11107806 sha256=f6a5416e2147d8a058a2e5c18d3c3d0bcc083aa270cbfc47ccda50e0fdfd1e34
  Stored in directory: /tmp/pip-ephem-wheel-cache-eurrran5/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
!pip install -q peft  accelerate bitsandbytes safetensors

In [4]:
!pip install sentencepiece


### Model Initialization and Setup

In this section:
  
- *AutoModelForCausalLM*: provides an interface to load models designed for causal language modeling, i.e., predicting the next token in a sequence.


- `model_name`: Defines the identifier for the model we want to load. In this case, we're using the sharded version of the Mistral-7B model named [`"filipealmeida/Mistral-7B-Instruct-v0.1-sharded"`](https://huggingface.co/filipealmeida/Mistral-7B-Instruct-v0.1-sharded).


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"


### Setting up the BitsAndBytes Configuration

The code block below configures the `BitsAndBytes` quantization settings, which are designed to optimize model performance by reducing the memory requirements of the model parameters:

- `load_in_4bit`: This flag, set to `True`, instructs the model to load its weights in 4-bit quantization. This can reduce memory usage significantly, allowing for larger models to fit into memory.

- `bnb_4bit_use_double_quant`: When set to `True`, this flag enables double quantization, which can further enhance the efficiency of 4-bit quantization.

- `bnb_4bit_quant_type`: Specifies the type of 4-bit quantization to use. The value `"nf4"` represents a specific form of quantization, but details on this are needed for a more complete description.

- `bnb_4bit_compute_dtype`: This defines the data type to use for computations when the model weights are quantized. Here, `torch.bfloat16` is used, which is a 16-bit floating point representation that offers a balance between precision and memory usage.


In [6]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

### Loading the Pretrained Model with Quantization

The code below is responsible for loading our pretrained Mistral-7B model, utilizing the previously configured `BitsAndBytes` quantization settings:

- `model_name`: Specifies the identifier for the pretrained model we want to load, which we've previously set to the sharded version of the Mistral-7B model.

- `load_in_4bit`: With this set to `True`, the model loads its weights using 4-bit quantization, which significantly reduces memory requirements.

- `torch_dtype`: Specifies the data type for tensor computations. We've set it to `torch.bfloat16` to strike a balance between memory efficiency and computational precision.

- `quantization_config`: We provide the `BitsAndBytes` configuration (`bnb_config`) established in the previous step to apply the specified quantization settings during model loading.

By leveraging these settings, the model is loaded in a memory-optimized manner, ensuring that even large models like Mistral-7B can be effectively used in constrained environments.


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Tokenizer Initialization and Configuration

1. **Initialize the Tokenizer**: Using the `AutoTokenizer` class from the `transformers` library, we initialize a tokenizer corresponding to our predefined model, `model_name`.
2. **Set Beginning of Sequence Token**: The `bos_token_id` is set to `1`, designating this token ID as the beginning of a sequence.
3. **Define Stop Tokens**: We define a list of token IDs, `stop_token_ids`, that signify stopping points in token sequences. Here, the token ID `0` is considered a stop token.
4. **Confirmation Print**: A print statement confirms the successful loading of the model into memory.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.bos_token_id = 1
stop_token_ids = [0]

print(f"Successfully loaded the model {model_name} into memory")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

Successfully loaded the model filipealmeida/Mistral-7B-Instruct-v0.1-sharded into memory


In [12]:
text = "Write some lyrics in the style of david bowie"

device = model.device  # get the device of the model
encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False)
# Move all tensors in the encoded dictionary to the same device
encoded = {k: v.to(device) for k, v in encoded.items()}

generated_ids = model.generate(**encoded, max_new_tokens=200, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Write some lyrics in the style of david bowie

Verse 1:
I'm a freak, I'm an outcast
Living in a world that hates
I'm a rock star, I wear makeup
But to some, it's still a mistake

Chorus:
I'm an alien, I don't belong
I'm an aberration, I don't fit in with the throng
I'm a freak and I'm proud
I'm not just a man, I'm a god

Verse 2:
I'm a bisexual, I love men and women too
But in this world of black and white, it's not enough for me to do
I'm a musical genius, I compose and I play
But to some, it's still not enough, they never take the time to ask and they say

Chorus:
I'm an alien


In [11]:
text = "Write some lyrics in the style of david bowie"

device = model.device  # get the device of the model
encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False)
# Move all tensors in the encoded dictionary to the same device
encoded = {k: v.to(device) for k, v in encoded.items()}

generated_ids = model.generate(**encoded, max_new_tokens=200, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Write some lyrics in the style of david bowie:
"I'm a little boy with a big blue dream
A tiny star in a vast, cold room
I wear a rainbow coat of hope and love
And I'm gonna change the world
Oh, everybody's talking,
But I won't listen to their lies
I'll dance to my own beat
And I'll paint my own skies
I'll rise above the darkness
And I'll find my own light
I'll be a star, a shooting star
Shining bright and pure and bright
Oh, I won't give up, oh I won't give in
I'll keep on going, through all the pain
I'll keep on fighting, till the end
Cause I won't give up, no
Cause I won't give up, no
I'll keep on fighting, till I'm born
Cause I won't give
